#### vision : 모델이 이미지를 입력으로 받아 이해하는 능력
- 사진설명, 영수증 분석, 표 읽기, 차트 해석...
- 전용모델 : gpt-image-1 (이미지 이해, 생성)

- Response API
- Images API
- Chat Completions API : 예전 방식

In [1]:
from dotenv import load_dotenv, find_dotenv
# .env 파일 정보 가져오기
load_dotenv(find_dotenv())

True

In [ ]:
# 웹에 있는 특정 이미지 해석
# https://img8.yna.co.kr/etc/inner/KR/2022/06/05/AKR20220605027200005_03_i_P4.jpg
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5-mini",
    input=[
     {   "role":"user",
        "content":[
            {"type":"input_text", "text":"이 이미지에 무엇이 보이나요?"},
            {"type":"input_image", "image_url":"https://img8.yna.co.kr/etc/inner/KR/2022/06/05/AKR20220605027200005_03_i_P4.jpg"},

        ]}
    ]
)

print(response.output_text)

In [4]:
# 로컬 파일 해석
with open("./image/bag.png", "rb") as fr:
    file = client.files.create(file=fr, purpose="vision")

print("업로드된 파일 ID ", file.id)

업로드된 파일 ID  file-SzHhZ38GDxopbDt97SZRMx


In [5]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini", # vision 지원
    input=[
     {   "role":"user",
        "content":[
            {"type":"input_text", "text":"이 상품 이미지를 보고 쇼핑몰 설명을 작성해줘"},
            {"type":"input_image", "file_id":file.id},

        ]}
    ]
)

print(response.output_text)

### 상품 설명: BEICHAO 오버사이즈 더플백

신뢰할 수 있는 스타일과 실용성을 갖춘 BEICHAO 오버사이즈 더플백을 소개합니다. 

- **섬세한 디자인**: 고급스러운 올리브 그린 색상과 세련된 브라운 스트랩이 조화를 이루어 어디에나 잘 어울립니다.
- **넉넉한 수납공간**: 넓은 본체와 다용도로 사용할 수 있는 지퍼 포켓이 있어 소지품을 효율적으로 정리할 수 있습니다.
- **편리한 휴대성**: 짐을 쉽게 옮길 수 있도록 디자인된 핸들과 조절 가능한 숄더 스트랩으로 다양한 스타일링이 가능합니다.
- **내구성**: 견고한 소재로 제작되어 오랜 사용에도 문제 없습니다.

여행, 운동, 일상 등 다양한 상황에서 활용할 수 있는 이 더플백으로 당신의 매력을 더해보세요!


In [6]:
# 이미지 생성
from openai import OpenAI
import base64
client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini", # vision 지원
    input="Generate an image of gray tabby cat hugging an otter with an orange scarf",
    tools=[{"type":"image_generation"}]
  
)
# 파일 저장
image_data = [output.result for output in response.output if output.type == "image_generation_call"]

if image_data:
    image_base64 = image_data[0]
    with open("./image/cat_and_otter.png", "wb") as f:
        f.write(base64.b64decode(image_base64))

print(response.output_text)

In [ ]:
# !pip install pillow


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# 이미지 생성 전용 모델
from openai import OpenAI
import base64
from io import BytesIO
from PIL import Image
client = OpenAI()

response = client.images.generate(
    model="gpt-image-1-mini",
    prompt="A cute red panda reading a book, watercolor style",
    size="1024x1024",
    quality="high",
    output_format="png"
)
# 파일 저장
b64 = response.data[0].b64_json
img_bytes = base64.b64decode(b64)
img = Image.open(BytesIO(img_bytes)).convert("RGB")
img.save("./image/panda.png")